In [ ]:
import pickle
import sys
import CRPS.CRPS as pscore
import numpy as np
from pathlib import Path

import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TORCH_NUM_THREADS"] = "1"

import multiprocessing as mp
mp.set_start_method('spawn')

sys.path.insert(0, '../LSTM_next_activity_duration/notebooks/evaluation/')
sys.path.insert(0, '../../../../Evaluation')

import conduct_evaluation
import normal_evaluation.normal_evaluation
from prefix_duration_predictor import PrefixDurationPredictor, NOTEBOOK_DIR
from normal_evaluation.lstm_evaluation import SampleOutcomes_LSTM


get_pscores = lambda likelihoods : [pscore(likelihoods[1][i], likelihoods[2][k][3]).compute()[0] for i, k in enumerate(list(likelihoods[0].keys()))]

In [ ]:
with open('../../../transformed_event_logs/BPIC_19_test.pickle', 'rb') as f:
    test_data = pickle.load(f)


n_processes = 32
batch_size = 5
N = 1000

In [ ]:
event_log_properties = {
    'case_name' : 'case:concept:name',
    'concept_name' : 'concept:name_start',
    'timestamp_name' : 'time:timestamp_start',
    'time_since_case_start_column' : '',
    'time_since_last_event_column' : '',
    'day_in_week_column' : 'day_in_week',
    'seconds_in_day_column' : 'seconds_in_day',
    'min_suffix_size' : 1,
    'train_validation_size' : 0.15,
    'test_validation_size' : 0.0,
    'window_size' : 'auto',
    'categorical_columns' : ['concept:name_start', 'org:resource_start'],
    'continuous_columns' : ['seconds_in_day', 'day_in_week', 'duration_seconds'],
    'continuous_positive_columns' : []
}

#NOTEBOOK_DIR = Path(__file__).resolve().parent
LSTM_ROOT = (NOTEBOOK_DIR / "../..").resolve()
LOADER_DIR = (NOTEBOOK_DIR / "../../../../load/event_log_loader").resolve()
ENCODED_DIR = (NOTEBOOK_DIR / "../../../../load/encoded_data").resolve()
TRANSFORMED_LOG_DIR = (NOTEBOOK_DIR / "../../../../../transformed_event_logs").resolve()
MODEL_DIR = (NOTEBOOK_DIR / "../training_variational_dropout/BPIC19").resolve()

TRAIN_DATA_PATH = (ENCODED_DIR / "BPIC_2019_all_1_train.pkl").resolve()

selected_cat_attributes = ['concept:name_start', 'org:resource_start']
selected_num_attributes = ['seconds_in_day', 'day_in_week']

lstm_predictor = PrefixDurationPredictor(
        train_loader_path = TRAIN_DATA_PATH,
        model_dir = MODEL_DIR,
        model_path = None,
        event_log_properties= event_log_properties,
        selected_cat_attributes = selected_cat_attributes,
        selected_num_attributes = selected_num_attributes,
        device = 'cpu'
)

from pyinstrument import Profiler

prof = Profiler()
prof.start()
try:
    evaluator_A = conduct_evaluation.ConductEvaluation(lstm_predictor, SampleOutcomes_LSTM, {
                                                        },
                                        test_data, n_processes=n_processes, batch_size=batch_size, n=N)
    likelihoods_A = evaluator_A.sample_cases(False, False)
except KeyboardInterrupt:
    print('interrupted - stopping profiling')
finally:
    prof.stop()
    html = prof.output_html()

    # save locally on remote
    with open("profile.html", "w") as f:
        f.write(html)

In [ ]:
evaluator_A = conduct_evaluation.ConductEvaluation(lstm_predictor, SampleOutcomes_LSTM, {
                                                    },
                                    test_data, n_processes=n_processes, batch_size=batch_size, n=N)
likelihoods_A = evaluator_A.sample_cases(False, True, False)

In [ ]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

In [ ]:
np.mean(get_pscores(likelihoods_A))